In [ ]:
# Make backend modules importable from either the backend or notebook directory.
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
backend_root = cwd / "backend" if (cwd / "backend" / "tools").is_dir() else cwd
backend_root = backend_root.parent if backend_root.name == "notebook" else backend_root
sys.path.insert(0, str(backend_root)) if str(backend_root) not in sys.path else None

from pm4py.objects.dcr.importer import importer as dcr_importer
from tools.find_relevant_laws import FindRelevantLaws
from tools.find_similar_cases import FindSimilarCases
from tools.interpret_input import InterpretInput
from tools.interpret_output import InterpretOutput
from tools.summarize_case import SummarizeCaseHistory

In [ ]:
# Retrieve locally, then ask the configured LLM for a cited answer.
law_query = "What requirements apply to compensation for necessary additional expenses?"
laws = FindRelevantLaws()
raw_laws = laws.find(law_query, top_k=5)
for result in raw_laws:
    print(result.source, result.page_number + 1, round(result.score, 3))
law_answer = await laws.answer(law_query, top_k=5)
print(law_answer)

In [ ]:
# Compare the closest cases and their outcome clusters.
case_query = "The child is under 18 and has a permanent disability, but expense documentation is incomplete."
cases = FindSimilarCases()
closest_cases = cases.find(case_query, top_k=5)
case_clusters = cases.cluster(case_query, top_k_per_outcome=3)
print([case.source for case in closest_cases])
print("positive", [case.source for case in case_clusters.positive])
print("negative", [case.source for case in case_clusters.negative])
case_answer = await cases.answer(case_query, top_k=5, top_k_per_outcome=3)
print(case_answer)

In [ ]:
# Combine the current record with locally retrieved law and cases.
model_path = backend_root / "data" / "models" / "Social Service Law 86 Data EN.xml"
graph = dcr_importer.apply(model_path, variant=dcr_importer.DCR_JS_PORTAL)
history = [
    {"role": "Citizen", "message": "The child is 12 and has a permanent disability."},
    {"role": "Citizen", "message": "Some expense receipts are still missing."},
]
trace = ["Age supplied", "Disability described", "Expenses partly documented"]
summary = await SummarizeCaseHistory().get_summary(
    history, trace, raw_laws, closest_cases, graph
)
print(summary)

In [ ]:
# Interpret a typed answer and generate a question for its DCR activity.
age_activity = graph.getActivity("Event_07k41iu")
interpreted_age = await InterpretInput().get_closest_match(
    "The child is twelve years old", int
)
question = await InterpretOutput().get_question(age_activity, graph)
print(interpreted_age, type(interpreted_age))
print(question)